# 🎬 Аналитический отчет MovieLens


В данном отчете мы проводим анализ набора данных MovieLens (первые 1000 записей) с целью выявления трендов в киноиндустрии, 
изучения предпочтений пользователей и оценки экономического успеха фильмов.
Для анализа используется специально разработанный Python-модуль movielens_analysis, оптимизированный для работы без использования
сторонних библиотек (таких как Pandas или NumPy).

## 🛠 Инициализация
Первым делом инициализируем наши аналитические классы и загрузим данные. Мы также использовали самописные функции для красоты отчета.


In [1]:
import movielens_analysis as ma


path_movies = '../datasets/movies.csv'
path_ratings = '../datasets/ratings.csv'
path_tags = '../datasets/tags.csv'
path_links = '../datasets/links.csv'

movies = ma.Movies(path_movies)
ratings = ma.Ratings(path_ratings, path_movies)
tags = ma.Tags(path_tags)
links = ma.Links(path_links, path_movies)

## Глава 1. История кино 🎥

Начнем с анализа самих фильмов. Когда они были выпущены? Какие жанры наиболее популярны?

### 1.1. Годы выпуска
Как распределены фильмы по годам выпуска? Мы ожидаем увидеть преобладание более современных картин.

In [2]:
%timeit -n 10 -r 3 movies.dist_by_release()
ma.plot_ascii_bar(movies.dist_by_release())

2.06 ms ± 464 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
1995                 | █████████████████████████ (224)
1994                 | ████████████████████ (184)
1996                 | ████████████████████ (181)
1993                 | ███████████ (101)
1992                 | ██ (23)
1990                 | █ (15)
1991                 | █ (15)
1989                 | █ (14)
1986                 | █ (9)
1982                 |  (8)
1940                 |  (8)
1957                 |  (8)
1987                 |  (8)
1980                 |  (8)
1981                 |  (7)
--------------------------------------------------


**💡 Инсайт: Эффект «Актуального кино»**

График демонстрирует экстремальный перекос в сторону середины 90-х годов: **более 60%** всех фильмов в выборке приходятся всего на три года (1994–1996).

Это не означает, что в 90-е снимали в 20 раз больше фильмов, чем в 80-е. Мы наблюдаем **артефакт создания базы данных**:
1.  Поскольку мы анализируем первые 1000 ID, мы видим самые первые записи, добавленные в MovieLens (проект стартовал в 1997 году).
2.  В первую очередь в базу добавляли "свежие" на тот момент хиты (1995–1996) и культовую классику прошлых лет.
3.  Фильмы 1940–1980 годов представлены точечно — это "золотой фонд", который был оцифрован в первую очередь.

**Вывод:** Наша выборка репрезентует не всю историю кино, а **срез поп-культуры середины 90-х**.

### 1.2. Жанры
Какие жанры доминируют в нашей выборке?

In [3]:
%timeit -n 10 -r 3 movies.dist_by_genres()
ma.plot_ascii_bar(movies.dist_by_genres())

2.16 ms ± 410 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
Drama                | █████████████████████████ (507)
Comedy               | █████████████████ (365)
Romance              | ██████████ (208)
Thriller             | ████████ (179)
Action               | ███████ (158)
Adventure            | ██████ (126)
Crime                | ██████ (122)
Children             | ████ (100)
Fantasy              | ███ (69)
Sci-Fi               | ███ (69)
Mystery              | ██ (58)
Musical              | ██ (53)
Horror               | ██ (51)
War                  | ██ (48)
Animation            | █ (37)
--------------------------------------------------


**💡 Инсайт: Жанровая универсальность**

Мы видим абсолютное доминирование **Драмы (507)** и **Комедии (365)**.
1.  **Драма как «клей»:** То, что у половины фильмов есть тег "Drama", говорит о том, что это не столько самостоятельный жанр в этой выборке, сколько универсальная составляющая. Фильмы часто маркируются как `Crime|Drama`, `War|Drama` или `Romance|Drama`.
2.  **Триллер обошел Экшен:** Интересно, что **Триллер (179)** популярнее, чем **Экшен (158)**. Это характерно для кино 90-х, когда психологические и детективные триллеры («Семь», «Молчание ягнят») были на пике моды, в то время как CGI-блокбастеры только набирали обороты.
3.  **Мультижанровость:** Сумма всех значений намного превышает 1000 (количество фильмов). Это подтверждает, что «чистых» жанров почти не снимают — большинство картин находятся на стыке категорий.

### 1.3. Сложность жанров
Какие фильмы являются наиболее "сложными" с точки зрения жанровой принадлежности (сочетают в себе наибольшее количество жанров)?

In [4]:
%timeit -n 10 -r 3 movies.most_genres(10)
ma.print_top(movies.most_genres(10))

2.02 ms ± 325 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

🏆 Топ
1. Strange Days (1995)            | 6 
2. Lion King, The (1994)          | 6 
3. Getaway, The (1994)            | 6 
4. Super Mario Bros. (1993)       | 6 
5. Beauty and the Beast (1991)    | 6 
6. All Dogs Go to Heaven 2 (1996) | 6 
7. Space Jam (1996)               | 6 
8. Aladdin and the King of Thieves (1996) | 6 
9. Toy Story (1995)               | 5 
10. Money Train (1995)             | 5 


**💡 Инсайт: Феномен «Диснеевского набора»**

Удивительно, но список самых жанрово-насыщенных фильмов возглавляют не сложные артхаусные драмы, а **мультипликация и семейное кино** (*The Lion King, Beauty and the Beast, Space Jam, Super Mario Bros*).

Это объясняется спецификой категоризации семейных фильмов. Чтобы охватить максимально широкую аудиторию, такие картины одновременно получают множество тегов.

**Вывод:** В базе данных MovieLens «сложность» жанров часто указывает не на глубину сюжета, а на **маркетинговую стратегию** фильма — попытку понравиться всем сразу (и детям, и родителям)

### 1.4. Жанровые тренды по десятилетиям ⏳
Меняются ли вкусы зрителей со временем? Этот метод группирует фильмы по десятилетиям и выделяет топ-5 самых популярных жанров для каждого периода.
Это позволяет проследить эволюцию индустрии: например, золотую эру вестернов, расцвет боевиков в 90-х или бум супергеройского кино в 2010-х.

In [5]:
%timeit -n 10 -r 3 movies.top_genres_by_decade()
trends = movies.top_genres_by_decade()
for decade in sorted(trends.keys()):
    print(f"{decade}: {trends[decade]}")

2.59 ms ± 580 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)
1930s: {'Comedy': 10, 'Romance': 10, 'Drama': 9, 'Musical': 6, 'Children': 4}
1940s: {'Drama': 16, 'Romance': 10, 'Mystery': 9, 'Thriller': 7, 'Film-Noir': 7}
1950s: {'Drama': 26, 'Romance': 15, 'Comedy': 12, 'Adventure': 10, 'Musical': 7}
1960s: {'Drama': 19, 'Comedy': 12, 'Romance': 9, 'Children': 8, 'Musical': 7}
1970s: {'Drama': 18, 'Comedy': 17, 'Crime': 7, 'Children': 7, 'Adventure': 5}
1980s: {'Drama': 37, 'Action': 22, 'Comedy': 21, 'Adventure': 16, 'Sci-Fi': 14}
1990s: {'Drama': 382, 'Comedy': 287, 'Romance': 148, 'Thriller': 147, 'Action': 126}


**💡 Инсайт: От эскапизма к эре блокбастеров**

Данные показывают, как менялся «дух времени» в киноиндустрии:

1.  **30-е и 40-е (Золотой век):** В 30-е лидируют легкие жанры (**Comedy, Romance, Musical**) — кино было способом отвлечься от Великой депрессии. В 40-е настроение меняется: в топ выходят **Mystery** и **Film-Noir**, отражая тревожную атмосферу военного и послевоенного времени.
2.  **70-е (Новый Голливуд):** Впервые в топе появляется жанр **Crime** (Криминал). Это эпоха реалистичного, жесткого кино («Крестный отец», «Таксист»).
3.  **80-е (Переломный момент):** Происходит фундаментальный сдвиг. Исчезают мюзиклы, а на их место приходят **Action** и **Sci-Fi**. Это рождение эры современных блокбастеров (эффект Лукаса и Спилберга).
4.  **90-е:** Триллер (**Thriller**) становится одним из главных жанров, практически догоняя романтику.
5.  В целом большую часть времени жанр Драма остается лидером

**Вывод:** Мы видим, как кино прошло путь от театральных жанров (мюзикл, драма) к технологичным и зрелищным (фантастика, боевик).

## Глава 2. Критики (Рейтинги) ⭐
Теперь посмотрим, как пользователи оценивают эти фильмы.

### 2.1. Рейтинги по годам
Оценивают ли пользователи новинки чаще, чем старую классику?

In [6]:
%timeit -n 10 -r 3 ratings.movies.dist_by_year()
ma.plot_ascii_bar(ratings.movies.dist_by_year())

471 μs ± 27.5 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
1996                 | █████████████████████████ (358)
1999                 | █████ (82)
2000                 | ████████████████████ (296)
2001                 | ████ (70)
2005                 | ████████ (121)
2006                 |  (4)
2007                 |  (1)
2011                 | ██ (39)
2015                 | ██ (29)
--------------------------------------------------


**💡 Инсайт: Волны активности аудитории**

График распределения рейтингов по времени показывает не плавный рост, а **дискретные всплески активности**:

1.  **1996 год (358 оценок):** Это «первая волна». MovieLens был запущен в 1997 году (на базе GroupLens), но данные начали собираться чуть раньше. Пользователи массово регистрировались и оценивали всё, что посмотрели за прошлые годы (бэк-каталог).
2.  **2000 и 2005 годы:** Мы видим четкие новые пики. Это указывает на то, что наша выборка (первые 1000 строк) содержит данные конкретных пользователей, которые были активны именно в эти периоды.

### 2.2. Распределение оценок
Какая оценка встречается чаще всего? Склонны ли люди ставить "пятерки" или они более критичны?

In [7]:
%timeit -n 10 -r 3 ratings.movies.dist_by_rating()
ma.plot_ascii_bar(ratings.movies.dist_by_rating())

82.6 μs ± 10.1 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
0.5                  | ██ (24)
1.0                  | ███ (39)
1.5                  |  (11)
2.0                  | ████ (57)
2.5                  |  (7)
3.0                  | █████████████████████ (253)
3.5                  | █ (17)
4.0                  | █████████████████████████ (292)
4.5                  | ██ (33)
5.0                  | ██████████████████████ (267)
--------------------------------------------------


**Самые частые оценки это 3, 4, 5**

### 2.3. Самые популярные фильмы
Какие фильмы имеют наибольшее количество оценок? Это, вероятно, главные блокбастеры выборки.

In [8]:
%timeit -n 10 -r 3 ratings.movies.top_by_num_of_ratings(10)
ma.print_top(ratings.movies.top_by_num_of_ratings(10))

193 μs ± 26.1 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

🏆 Топ
1. Usual Suspects, The (1995)     | 4 
2. Pulp Fiction (1994)            | 4 
3. Fugitive, The (1993)           | 4 
4. Schindler's List (1993)        | 4 
5. Batman (1989)                  | 4 
6. Silence of the Lambs, The (1991) | 4 
7. Fargo (1996)                   | 4 
8. Aladdin (1992)                 | 4 
9. Beauty and the Beast (1991)    | 4 
10. Toy Story (1995)               | 3 


**Культовые фильмы!!** Однако из-за того, что мы анализируем только первые 1000 строк, выборка нелостаточна для полного анализа

### 2.4. Фильмы с лучшим рейтингом
Найдем скрытые шедевры. Посмотрим топ-10 фильмов по средней оценке.

In [9]:
%timeit -n 10 -r 3 ratings.movies.top_by_ratings(10)
ma.print_top(ratings.movies.top_by_ratings(10))

983 μs ± 124 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

🏆 Топ
1. Bottle Rocket (1996)           | 5.00 
2. Canadian Bacon (1995)          | 5.00 
3. Star Wars: Episode IV - A New Hope (1977) | 5.00 
4. James and the Giant Peach (1996) | 5.00 
5. Wizard of Oz, The (1939)       | 5.00 
6. Citizen Kane (1941)            | 5.00 
7. Adventures of Robin Hood, The (1938) | 5.00 
8. Mr. Smith Goes to Washington (1939) | 5.00 
9. Winnie the Pooh and the Blustery Day (1968) | 5.00 
10. Three Caballeros, The (1945)   | 5.00 


 Из-за того, что мы анализируем только первые 1000 строк, выборка нелостаточна для полного анализа

А теперь посмотрим топ-10 по медианной оценке, чтобы исключить влияние редких выбросов.

In [10]:
%timeit -n 10 -r 3 ratings.movies.top_by_ratings(10, metric=ma.Ratings.Movies.median)
ma.print_top(ratings.movies.top_by_ratings(10, metric=ma.Ratings.Movies.median))

959 μs ± 91 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

🏆 Топ
1. Bottle Rocket (1996)           | 5.00 
2. Canadian Bacon (1995)          | 5.00 
3. Star Wars: Episode IV - A New Hope (1977) | 5.00 
4. Tommy Boy (1995)               | 5.00 
5. Forrest Gump (1994)            | 5.00 
6. Fugitive, The (1993)           | 5.00 
7. Jurassic Park (1993)           | 5.00 
8. Tombstone (1993)               | 5.00 
9. Dances with Wolves (1990)      | 5.00 
10. Pinocchio (1940)               | 5.00 


 Из-за того, что мы анализируем только первые 1000 строк, выборка нелостаточна для полного анализа

### 2.5 Самые противоричевые фильмы 
По каким фильмам самый большой разброс оценок?

In [11]:
%timeit -n 10 -r 3 ratings.movies.top_controversial(10)
ma.print_top(ratings.movies.top_controversial(10))

1.34 ms ± 179 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

🏆 Топ
1. Bambi (1942)                   | 5.06 
2. Rescuers, The (1977)           | 5.06 
3. My Fair Lady (1964)            | 5.06 
4. Matrix, The (1999)             | 4.00 
5. Schindler's List (1993)        | 3.42 
6. Good Will Hunting (1997)       | 3.06 
7. Courage Under Fire (1996)      | 3.06 
8. Usual Suspects, The (1995)     | 2.42 
9. Dazed and Confused (1993)      | 2.25 
10. Fight Club (1999)              | 2.25 


 Из-за того, что мы анализируем только первые 1000 строк, выборка нелостаточна для полного анализа

## Глава 3. Аудитория (Пользователи) 👤
Кто те люди, которые ставят эти оценки?

### 3.1. Самые активные пользователи
Кто смотрит и оценивает больше всего фильмов?

In [12]:
%timeit -n 10 -r 3 ratings.users.top_by_num_of_ratings(10)
ma.plot_ascii_bar(ratings.users.top_by_num_of_ratings(10))

51.7 μs ± 6.58 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
6                    | █████████████████████████ (314)
1                    | ██████████████████ (232)
4                    | █████████████████ (216)
7                    | ██████████ (126)
5                    | ███ (44)
3                    | ███ (39)
2                    | ██ (29)
--------------------------------------------------


**Некоторые пользователи во много раз активнее других**

### 3.2. Характер критиков
Кто является самым строгим критиком (низкий средний балл), а кто — самым добрым зрителем?

In [13]:
%timeit -n 10 -r 3 ratings.users.top_by_ratings(10, metric=ma.Ratings.Movies.average)
ma.plot_ascii_bar(ratings.users.top_by_ratings(10, metric=ma.Ratings.Movies.average))

97.4 μs ± 11.4 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
1                    | █████████████████████████ (4.37)
2                    | ██████████████████████ (3.95)
5                    | ████████████████████ (3.64)
4                    | ████████████████████ (3.56)
6                    | ███████████████████ (3.49)
7                    | ███████████████████ (3.35)
3                    | █████████████ (2.44)
--------------------------------------------------


Как мы видим есть только один критик, который ставит оценку два. Большинство предпочитают ставить оценку 4

### 3.3. Непоследовательные пользователи
У каких пользователей самый нестабильный вкус (высокая дисперсия оценок)?

In [14]:
%timeit -n 10 -r 3 ratings.users.top_controversial(10)
ma.plot_ascii_bar(ratings.users.top_controversial(10))

240 μs ± 49.8 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
3                    | █████████████████████████ (4.26)
4                    | ██████████ (1.72)
7                    | █████████ (1.65)
5                    | █████ (0.96)
6                    | ████ (0.72)
1                    | ███ (0.64)
2                    | ███ (0.63)
--------------------------------------------------


Критик, который ставит самую низкую среднюю оценку имеет самый большой разброс в своих оценках.

## Глава 4. Анализ метаданных (Теги) 🏷️
Что люди пишут о фильмах в тегах? Это дает нам ключевые слова для понимания контента.

### 4.1. Самые популярные теги
Какие слова используются чаще всего?

In [15]:
%timeit -n 10 -r 3 tags.most_popular(10)
ma.plot_ascii_bar(tags.most_popular(10))

70.3 μs ± 4.14 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
funny                | █████████████████████████ (15)
sci-fi               | ███████████████████████ (14)
twist ending         | ████████████████████ (12)
dark comedy          | ████████████████████ (12)
atmospheric          | ████████████████ (10)
superhero            | ████████████████ (10)
comedy               | ████████████████ (10)
action               | ████████████████ (10)
suspense             | ████████████████ (10)
Leonardo DiCaprio    | ███████████████ (9)
--------------------------------------------------


*funny и sci-fi* в лидерах

### 4.2. Анализ уникальных тегов
Найдем самые длинные теги и теги с наибольшим количеством слов.

In [16]:
%timeit -n 10 -r 3 tags.longest(10)
print("Самые длинные теги:")
for i in tags.longest(10):
    print(i)

55.8 μs ± 1.34 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)
Самые длинные теги:
Something for everyone in this one... saw it without and plan on seeing it with kids!
the catholic church is the most corrupt organization in history
audience intelligence underestimated
Oscar (Best Music - Original Score)
assassin-in-training (scene)
Everything you want is here
political right versus left
Oscar (Best Cinematography)
representation of children
artificial intelligence


In [17]:
%timeit -n 10 -r 3 tags.most_words(10)
ma.plot_ascii_bar(tags.most_words(10))

497 μs ± 42.6 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
Something for everyo | █████████████████████████ (16)
the catholic church  | ███████████████ (10)
based on a true stor | ███████ (5)
Everything you want  | ███████ (5)
Oscar (Best Music -  | ███████ (5)
based on a play      | ██████ (4)
political right vers | ██████ (4)
lord of the rings    | ██████ (4)
seen more than once  | ██████ (4)
assassin-in-training | ██████ (4)
--------------------------------------------------


Пересечение: теги, которые одновременно длинные и многословные.

In [18]:
%timeit -n 10 -r 3 tags.most_words_and_longest(10)
for i in tags.most_words_and_longest(10):
    print(i)

658 μs ± 55.9 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)
political right versus left
Everything you want is here
the catholic church is the most corrupt organization in history
Oscar (Best Music - Original Score)
assassin-in-training (scene)
Something for everyone in this one... saw it without and plan on seeing it with kids!


### 4.3. Поиск по тегам
Давайте посмотрим, какие теги содержат слово "funny" (смешной).

In [19]:
%timeit -n 10 -r 3 tags.tags_with('funny')
print(tags.tags_with('funny'))

32.6 μs ± 8.88 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)
['funny', 'stupid but funny', 'very funny']


## Глава 5. Экономика и производство (IMDb) 💰
Наконец, мы объединяем данные MovieLens с данными IMDb (Бюджет, Сборы, Режиссеры), чтобы понять бизнес-составляющую кино.
Данные IMDb были предварительно собраны и закэшированы для ускорения анализа.

### 5.1. Топ режиссеров
Кто снял наибольшее количество фильмов, попавших в эту выборку?

In [20]:
%timeit -n 10 -r 3 links.top_directors(10)
ma.plot_ascii_bar(links.top_directors(10))

855 μs ± 37.6 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
Alfred Hitchcock     | █████████████████████████ (11)
Woody Allen          | ██████████████████ (8)
Frank Capra          | █████████████ (6)
Stanley Kubrick      | █████████████ (6)
Steven Spielberg     | ███████████ (5)
Rob Reiner           | ███████████ (5)
James Cameron        | ███████████ (5)
Martin Scorsese      | ███████████ (5)
John Carpenter       | ███████████ (5)
Tony Scott           | ███████████ (5)
--------------------------------------------------


**Хичкок лидер**, а пытается догнать его другая легенда, что неудивительно

### 5.2. Самые дорогие фильмы
Какие фильмы имели самый высокий производственный бюджет?

In [21]:
%timeit -n 10 -r 3 links.most_expensive(10)
ma.plot_ascii_bar(links.most_expensive(10))

3.45 ms ± 645 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
Waterworld (1995)    | █████████████████████████ (175,000,000.00)
True Lies (1994)     | ████████████████ (115,000,000.00)
Terminator 2: Judgme | ██████████████ (102,000,000.00)
Eraser (1996)        | ██████████████ (100,000,000.00)
Hunchback of Notre D | ██████████████ (100,000,000.00)
Batman Forever (1995 | ██████████████ (100,000,000.00)
Cutthroat Island (19 | ██████████████ (98,000,000.00)
Twister (1996)       | █████████████ (92,000,000.00)
Judge Dredd (1995)   | ████████████ (90,000,000.00)
Die Hard: With a Ven | ████████████ (90,000,000.00)
--------------------------------------------------


**Как мы видим, даже тогда бюджеты были значительные**

### 5.3. Самые прибыльные фильмы
Какие картины принесли наибольшую прибыль (Мировые сборы минус Бюджет)?

In [22]:
%timeit -n 10 -r 3 links.most_profitable(10)
ma.plot_ascii_bar(links.most_profitable(10))

5.37 ms ± 718 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
Jurassic Park (1993) | █████████████████████████ (1,041,381,981.00)
Lion King, The (1994 | ██████████████████████ (934,161,373.00)
E.T. the Extra-Terre | ██████████████████ (786,807,407.00)
Star Wars: Episode I | ██████████████████ (764,398,507.00)
Independence Day (a. | █████████████████ (742,400,891.00)
Forrest Gump (1994)  | ██████████████ (623,226,465.00)
Star Wars: Episode V | ████████████ (532,016,086.00)
Ghost (1990)         | ███████████ (483,703,557.00)
Aladdin (1992)       | ███████████ (476,050,219.00)
Home Alone (1990)    | ███████████ (458,684,675.00)
--------------------------------------------------


**Jurassic Park (1993)** даже на сегодняшний день очень внушительные сборы

### 5.4. Самые длинные фильмы
У каких фильмов самый большой хронометраж?

In [23]:
%timeit -n 10 -r 3 links.longest(10)
ma.plot_ascii_bar(links.longest(10))

3.1 ms ± 690 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
Madonna: Truth or Da | █████████████████████████ (240)
Jean de Florette (19 | █████████████████████████ (240)
Platoon (1986)       | █████████████████████████ (240)
William Shakespeare' | █████████████████████████ (240)
Get on the Bus (1996 | █████████████████████████ (240)
Foreign Corresponden | █████████████████████████ (240)
Emma (1996)          | █████████████████████████ (240)
Gone with the Wind ( | ████████████████████████ (238)
Once Upon a Time in  | ███████████████████████ (229)
Lawrence of Arabia ( | ███████████████████████ (227)
--------------------------------------------------


**Как правило фильмы не превышают 4 часов**

### 5.5. Стоимость минуты
Какие фильмы были самыми дорогими в производстве в пересчете на одну минуту экранного времени?

In [24]:
%timeit -n 10 -r 3 links.top_cost_per_minute(10)
ma.plot_ascii_bar(links.top_cost_per_minute(10))

5.88 ms ± 921 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
Waterworld (1995)    | █████████████████████████ (1,296,296.30)
Hunchback of Notre D | █████████████████████ (1,098,901.10)
Judge Dredd (1995)   | ██████████████████ (937,500.00)
Space Jam (1996)     | █████████████████ (909,090.91)
Eraser (1996)        | ████████████████ (869,565.22)
Batman Forever (1995 | ███████████████ (826,446.28)
True Lies (1994)     | ███████████████ (815,602.84)
Twister (1996)       | ███████████████ (814,159.29)
Cutthroat Island (19 | ███████████████ (790,322.58)
Terminator 2: Judgme | ██████████████ (744,525.55)
--------------------------------------------------


**Как мы видим Waterworld был один из самых масштабных фильмов того времени**

### 5.6. Эффективность режиссеров (ROI) [Бонус] 📈
Окупаемость инвестиций (ROI) показывает, какие режиссеры работают наиболее эффективно.
В отличие от показателя «Самые прибыльные», где лидируют блокбастеры с огромными бюджетами, ROI позволяет выделить режиссеров, способных создавать коммерчески успешные фильмы при ограниченных ресурсах.

In [25]:
%timeit -n 10 -r 3 links.roi_by_director(10)
ma.plot_ascii_bar(links.roi_by_director(10))

6.43 ms ± 1.38 ms per loop (mean ± std. dev. of 3 runs, 10 loops each)

📊 График
--------------------------------------------------
Rose Troche          | █████████████████████████ (159.35)
"William Cottrell    | ███████████████████ (122.48)
David Hand           | ███████████████████ (122.48)
Victor Fleming       | ███████████████ (100.18)
George Lucas         | ██████████ (69.49)
Kevin Smith          | █████████ (57.56)
Wilfred Jackson"     | ████████ (55.83)
Wolfgang Reitherman  | ███████ (46.75)
"Norman Ferguson     | ███████ (45.88)
T. Hee               | ███████ (45.88)
--------------------------------------------------


**Rose Troche - режиссер с очень высоким ROI. Он приносит в среднем 159% прибыли относительно затрат**

## Заключение
В рамках данного отчета мы успешно проанализировали датасет MovieLens, используя кастомные классы Python. Мы изучили временное распределение релизов, поведение пользователей, теги и сопоставили эти данные с финансовыми показателями из IMDb.
Производительность каждого аналитического метода была измерена с помощью магической команды %timeit, что подтверждает эффективность реализованных алгоритмов.
